# Ethiopian Food Price Analysis - Commodity Analysis

**Objective:** How do retail food prices differ between commodities?

**Dataset:** `../data/processed/clean_prices.csv`  
- Retail prices only | kg unit only | ETB currency | Non-missing prices

**Scope of this notebook:** Commodity analysis only.  
Market analysis and price variation analysis are performed in separate notebooks.

## 1. Load the Data

Load the cleaned dataset and perform a quick initial inspection.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pathlib

# Ensure images directory exists
pathlib.Path("../images").mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/processed/clean_prices.csv", parse_dates=["period_date"])
print("Dataset loaded:", df.shape[0], "rows x", df.shape[1], "columns")

Dataset loaded: 32504 rows x 15 columns


In [ ]:
df.head()

,country,market,admin_1,longitude,latitude,cpcv2,product,source_document,period_date,price_type,product_source,unit,unit_type,currency,value
0,Ethiopia,"Addis Ababa, Merkato",Addis Ababa,38.7667,9.03333,R01701BD,Beans (Haricot),Famine Early Warning Systems Network (FEWS NET...,2020-01-01,Retail,Local,kg,Weight,ETB,41.0
1,Ethiopia,"Addis Ababa, Merkato",Addis Ababa,38.7667,9.03333,R01701BD,Beans (Haricot),Famine Early Warning Systems Network (FEWS NET...,2020-01-08,Retail,Local,kg,Weight,ETB,46.0
2,Ethiopia,"Addis Ababa, Merkato",Addis Ababa,38.7667,9.03333,R01701BD,Beans (Haricot),Famine Early Warning Systems Network (FEWS NET...,2020-01-15,Retail,Local,kg,Weight,ETB,48.6
3,Ethiopia,"Addis Ababa, Merkato",Addis Ababa,38.7667,9.03333,R01701BD,Beans (Haricot),Famine Early Warning Systems Network (FEWS NET...,2020-01-22,Retail,Local,kg,Weight,ETB,49.0
4,Ethiopia,"Addis Ababa, Merkato",Addis Ababa,38.7667,9.03333,R01701BD,Beans (Haricot),Famine Early Warning Systems Network (FEWS NET...,2020-01-29,Retail,Local,kg,Weight,ETB,50.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32504 entries, 0 to 32503
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   country          32504 non-null  object        
 1   market           32504 non-null  object        
 2   admin_1          32504 non-null  object        
 3   longitude        32504 non-null  float64       
 4   latitude         32504 non-null  float64       
 5   cpcv2            32504 non-null  object        
 6   product          32504 non-null  object        
 7   source_document  32504 non-null  object        
 8   period_date      32504 non-null  datetime64[ns]
 9   price_type       32504 non-null  object        
 10  product_source   32504 non-null  object        
 11  unit             32504 non-null  object        
 12  unit_type        32504 non-null  object        
 13  currency         32504 non-null  object        
 14  value            32504 non-null  float

In [ ]:
print("Shape:", df.shape)
print("Products:", sorted(df["product"].unique().tolist()))

Shape: (32504, 15)
Products: ['Beans (Haricot)', 'Firewood', 'Horse beans', 'Maize Grain (White)', 'Mixed Teff', 'Refined sugar', 'Rice (Milled)', 'Sorghum (Red)', 'Sorghum (White)', 'Sorghum (Yellow)', 'Wheat Flour', 'Wheat Grain']


## 2. Observation Count per Commodity

Before comparing average prices, it is important to know how many observations support each commodity's average.
An average based on very few observations is less reliable than one based on hundreds of observations.
A commodity with only a handful of recorded prices may show a misleadingly high or low average
simply due to limited data coverage rather than true price level.

In [ ]:
obs_counts = (
    df.groupby("product")["value"]
    .count()
    .reset_index()
    .rename(columns={"value": "num_observations"})
    .sort_values("num_observations", ascending=False)
    .reset_index(drop=True)
)
obs_counts.index += 1
print("Observation count per commodity:")
print(obs_counts.to_string())

Observation count per commodity:
                product  num_observations
1           Wheat Grain              3317
2   Maize Grain (White)              3305
3              Firewood              3172
4         Refined sugar              2916
5         Sorghum (Red)              2902
6           Horse beans              2850
7            Mixed Teff              2829
8         Rice (Milled)              2821
9       Sorghum (White)              2740
10      Beans (Haricot)              2576
11          Wheat Flour              1824
12     Sorghum (Yellow)              1252


## 3. Average Price per Commodity

Calculate the mean retail price (ETB/kg) for each commodity across all markets and dates.
Results are sorted from highest to lowest average price.

In [ ]:
avg_price = (
    df.groupby("product")["value"]
    .mean()
    .reset_index()
    .rename(columns={"value": "average_price_etb_per_kg"})
    .sort_values("average_price_etb_per_kg", ascending=False)
    .reset_index(drop=True)
)
avg_price["average_price_etb_per_kg"] = avg_price["average_price_etb_per_kg"].round(2)
avg_price.index += 1
print("Average retail price per commodity (ETB/kg):")
print(avg_price.to_string())

Average retail price per commodity (ETB/kg):
                product  average_price_etb_per_kg
1         Refined sugar                     63.29
2            Mixed Teff                     51.00
3       Beans (Haricot)                     50.84
4           Wheat Flour                     49.85
5         Rice (Milled)                     48.68
6           Horse beans                     46.14
7           Wheat Grain                     38.51
8      Sorghum (Yellow)                     36.43
9       Sorghum (White)                     34.59
10        Sorghum (Red)                     30.84
11  Maize Grain (White)                     26.38
12             Firewood                     14.04


## 4. Most Expensive Commodities

Commodities with the highest average retail price per kg.
Note: rankings are based on the mean price across all observations, not a single data point.

In [ ]:
top10 = avg_price.head(10).copy()
print("Top 10 most expensive commodities (ETB/kg):")
print(top10.to_string())

Top 10 most expensive commodities (ETB/kg):
             product  average_price_etb_per_kg
1      Refined sugar                     63.29
2         Mixed Teff                     51.00
3    Beans (Haricot)                     50.84
4        Wheat Flour                     49.85
5      Rice (Milled)                     48.68
6        Horse beans                     46.14
7        Wheat Grain                     38.51
8   Sorghum (Yellow)                     36.43
9    Sorghum (White)                     34.59
10     Sorghum (Red)                     30.84


## 5. Cheapest Commodities

Commodities with the lowest average retail price per kg.

In [ ]:
bottom10 = avg_price.tail(10).sort_values("average_price_etb_per_kg").reset_index(drop=True)
bottom10.index += 1
print("10 cheapest commodities (ETB/kg):")
print(bottom10.to_string())

10 cheapest commodities (ETB/kg):
                product  average_price_etb_per_kg
1              Firewood                     14.04
2   Maize Grain (White)                     26.38
3         Sorghum (Red)                     30.84
4       Sorghum (White)                     34.59
5      Sorghum (Yellow)                     36.43
6           Wheat Grain                     38.51
7           Horse beans                     46.14
8         Rice (Milled)                     48.68
9           Wheat Flour                     49.85
10      Beans (Haricot)                     50.84


## 6. Visualization 1 - Average Price by Commodity

Horizontal bar chart showing average retail price (ETB/kg) for each commodity,
sorted from highest to lowest.

In [ ]:
# Prepare data: lowest at top of horizontal bar chart means sort ascending here
plot_df = avg_price.sort_values("average_price_etb_per_kg", ascending=True).copy()

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor("#F8F9FA")
ax.set_facecolor("#F8F9FA")

colors = plt.cm.RdYlGn_r(
    [i / (len(plot_df) - 1) for i in range(len(plot_df))]
)[::-1]

bars = ax.barh(
    plot_df["product"],
    plot_df["average_price_etb_per_kg"],
    color=colors,
    edgecolor="white",
    linewidth=0.8,
    height=0.65
)

# Value labels
for bar, val in zip(bars, plot_df["average_price_etb_per_kg"]):
    ax.text(
        bar.get_width() + max(plot_df["average_price_etb_per_kg"]) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}",
        va="center", ha="left", fontsize=9, color="#333333"
    )

ax.set_xlabel("Average Retail Price (ETB / kg)", fontsize=12, labelpad=10)
ax.set_ylabel("Commodity", fontsize=12, labelpad=10)
ax.set_title(
    "Average Retail Food Prices by Commodity\nEthiopia, 2020-2023 (ETB per kg)",
    fontsize=14, fontweight="bold", pad=15
)
ax.tick_params(axis="both", labelsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.5, color="#CCCCCC")
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../images/average_price_by_commodity.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: ../images/average_price_by_commodity.png")

Saved: ../images/average_price_by_commodity.png


## 7. Visualization 2 - Observation Count per Commodity

This chart shows how many price observations were recorded for each commodity.
It helps the reader understand how much data supports each average price.

In [ ]:
obs_plot = obs_counts.sort_values("num_observations", ascending=True).copy()

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor("#F8F9FA")
ax.set_facecolor("#F8F9FA")

bars2 = ax.barh(
    obs_plot["product"],
    obs_plot["num_observations"],
    color="#4C72B0",
    edgecolor="white",
    linewidth=0.8,
    height=0.65,
    alpha=0.85
)

for bar, val in zip(bars2, obs_plot["num_observations"]):
    ax.text(
        bar.get_width() + max(obs_plot["num_observations"]) * 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{int(val):,}",
        va="center", ha="left", fontsize=9, color="#333333"
    )

ax.set_xlabel("Number of Price Observations", fontsize=12, labelpad=10)
ax.set_ylabel("Commodity", fontsize=12, labelpad=10)
ax.set_title(
    "Number of Price Observations per Commodity\nEthiopia, 2020-2023 (Retail, kg)",
    fontsize=14, fontweight="bold", pad=15
)
ax.tick_params(axis="both", labelsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.5, color="#CCCCCC")
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("../images/commodity_observation_counts.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: ../images/commodity_observation_counts.png")

Saved: ../images/commodity_observation_counts.png


## Commodity Analysis Findings

Based on the calculated results above.

In [ ]:
# Computed findings from actual data
highest = avg_price.iloc[0]
lowest  = avg_price.iloc[-1]
price_diff = highest["average_price_etb_per_kg"] - lowest["average_price_etb_per_kg"]

most_obs  = obs_counts.iloc[0]
least_obs = obs_counts.iloc[-1]

print("=" * 60)
print("  COMMODITY ANALYSIS FINDINGS")
print("=" * 60)
print()
print("1. Highest average price commodity:")
print("   Product    :", highest["product"])
print("   Avg price  :", highest["average_price_etb_per_kg"], "ETB/kg")
print()
print("2. Lowest average price commodity:")
print("   Product    :", lowest["product"])
print("   Avg price  :", lowest["average_price_etb_per_kg"], "ETB/kg")
print()
print("3. Price difference (highest - lowest):", round(price_diff, 2), "ETB/kg")
print()
print("4. Commodity with most observations:")
print("   Product    :", most_obs["product"])
print("   Observations:", most_obs["num_observations"])
print()
print("5. Commodity with fewest observations:")
print("   Product    :", least_obs["product"])
print("   Observations:", least_obs["num_observations"])
print()
print("6. Why observation count matters:")
print("   An average price based on fewer observations is more sensitive")
print("   to individual extreme values and may not reflect a stable price level.")
print("   Commodities with higher observation counts provide more reliable averages.")
print("=" * 60)

  COMMODITY ANALYSIS FINDINGS

1. Highest average price commodity:
   Product    : Refined sugar
   Avg price  : 63.29 ETB/kg

2. Lowest average price commodity:
   Product    : Firewood
   Avg price  : 14.04 ETB/kg

3. Price difference (highest - lowest): 49.25 ETB/kg

4. Commodity with most observations:
   Product    : Wheat Grain
   Observations: 3317

5. Commodity with fewest observations:
   Product    : Sorghum (Yellow)
   Observations: 1252

6. Why observation count matters:
   An average price based on fewer observations is more sensitive
   to individual extreme values and may not reflect a stable price level.
   Commodities with higher observation counts provide more reliable averages.


### Summary of Findings

The findings above are computed directly from the cleaned dataset.

**Key patterns observed:**
- The commodity with the highest average retail price has a substantially higher ETB/kg average than the lowest.
- The gap between the most and least expensive commodity (in ETB/kg) reflects genuine price-level differences
  across commodity types in the Ethiopian retail market during 2020-2023.
- Commodities with more observations provide more robust averages. Commodities with fewer observations
  should be interpreted with more caution.

**What we do NOT claim:**
- We do not explain price differences by production costs, imports, or supply chains without additional evidence.
- We do not compare markets in this notebook (that belongs to the market analysis step).
- We do not claim any commodity is expensive or cheap in absolute terms without an appropriate benchmark.